# Greedy highest-uncertainty controller

Candidate rectangles vote on whether every grid point is inside the hidden box. Binary entropy is highest where those candidates disagree most. This baseline repeatedly moves to the maximum-entropy grid point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from box_gym import BoxGym, action_toward, candidate_uncertainty

In [ ]:
def highest_uncertainty_goal(pred_boxes, grid_size=50):
    grid_x, grid_y, entropy = candidate_uncertainty(pred_boxes, grid_size)
    row, col = np.unravel_index(np.argmax(entropy), entropy.shape)
    goal = np.array([grid_x[row, col], grid_y[row, col]])
    return goal, (grid_x, grid_y, entropy)

In [ ]:
env = BoxGym(
    sensor_box_size=0.12,
    num_sensor_samples=4,
    max_velocity=0.25,
    inference_num=100,
)
obs, info = env.reset(seed=12)
trajectory = [obs["sensor_pos"].copy()]
goal, uncertainty_map = highest_uncertainty_goal(obs["pred_boxes"])

for step in range(300):
    # Replan as measurements change the candidate set.
    if step % 5 == 0 or np.linalg.norm(obs["sensor_pos"] - goal) < 0.025:
        goal, uncertainty_map = highest_uncertainty_goal(obs["pred_boxes"])

    action = action_toward(obs["sensor_pos"], goal, env.max_velocity)
    obs, reward, terminated, truncated, info = env.step(action)
    trajectory.append(obs["sensor_pos"].copy())
    if terminated or truncated:
        break

# Recompute the map for the final candidate set.
goal, uncertainty_map = highest_uncertainty_goal(obs["pred_boxes"])
print(f"Ran {step + 1} steps; final uncertainty: {info['uncertainty']:.5f}")

fig, ax = plt.subplots(figsize=(7, 7))
env.plot(
    ax,
    uncertainty_map=uncertainty_map,
    goal=goal,
    trajectory=np.asarray(trajectory),
)
ax.set_title("Greedy maximum-uncertainty policy")
plt.show()
env.close()